In [1]:
# ------------------------------
# 1️⃣ Create Spark Session with Delta Lake
# ------------------------------
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Mailchimp Delta Example") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")  # optional: reduce log spam

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6530aeeb-5f49-4078-927c-640476a1e03b;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (8081ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (485ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (722ms)
:: resolution report :: resolve 7043ms :: arti

In [2]:
# ------------------------------
# 2️⃣ Read CSV file
# ------------------------------
csv_path = "/home/jovyan/work/hubspot_processed_data.csv"  # exact path in Jupyter container

df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)
print("Schema of CSV:")
df.printSchema()
df.show(5)


Schema of CSV:
root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: double (nullable = true)
 |-- company: string (nullable = true)
 |-- lead_source: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- deal_value: double (nullable = true)
 |-- clv: double (nullable = true)
 |-- created_date: string (nullable = true)

+----------+---------+--------------------+-------------+--------------------+-----------+------------------+----------+--------+------------+
|first_name|last_name|               email|        phone|             company|lead_source|   lifecycle_stage|deal_value|     clv|created_date|
+----------+---------+--------------------+-------------+--------------------+-----------+------------------+----------+--------+------------+
|     Kayla|  Johnson|kayla.johnson45@g...|   6.71773E14|Duarte, Smith and...|   LinkedIn|        subscriber|   7485.72| 37428.6|  19-01-2024|


In [4]:
# ------------------------------
# 3️⃣ Convert to Delta Lake
# ------------------------------
delta_path = "/home/jovyan/work/delts_hubspot"  # folder to store delta table

df.write.format("delta").mode("overwrite").save(delta_path)

# ------------------------------
# 4️⃣ Read Delta Table to verify
# ------------------------------
df_delta = spark.read.format("delta").load(delta_path)
print("Delta Table Preview:")
df_delta.show(5)

25/09/22 12:46:50 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta Table Preview:
+----------+---------+--------------------+-------------+--------------------+-----------+------------------+----------+--------+------------+
|first_name|last_name|               email|        phone|             company|lead_source|   lifecycle_stage|deal_value|     clv|created_date|
+----------+---------+--------------------+-------------+--------------------+-----------+------------------+----------+--------+------------+
|     Kayla|  Johnson|kayla.johnson45@g...|   6.71773E14|Duarte, Smith and...|   LinkedIn|        subscriber|   7485.72| 37428.6|  19-01-2024|
|       Amy|  Dickson|amy.dickson12@gma...|   1.53967E13|        Vargas Group|   LinkedIn|          customer|   5387.22|21548.88|  25-05-2024|
|  Michelle|    Perez|michelle.perez162...|    1.3739E14|        Gonzales PLC|    Website|        subscriber|  12488.79|62443.95|  09-10-2023|
|    Sandra|   Larsen|sandra.larsen17@g...|4.220102762E9|        Taylor Group|   Referral|          customer|   10285.0| 